# 研究与工程思维 6/6：Pareto、风险与证据化决策

这不是一节“记术语”的课，而是一节**改变判断过程**的实验课。

| 项目 | 内容 |
|---|---|
| 核心问题 | 没有一个方案全面最好时，怎样做可解释、可复盘的选择？ |
| 迁移价值 | 适用于模型选型、上线门禁、资源规划、架构决策和个人学习路线。 |
| 建议投入 | 90～150 分钟；先预测，再运行，再保留被推翻的判断 |
| 通关证据 | 能把结论写成“主张—证据—反证—边界—下一步” |

固定闭环：

```text
观察（发生了什么） → 假设（可能为什么） → 区分性预测 → 最小实验
        ↑                                      ↓
        └──── 更新置信度、记录反例、决定下一步 ────┘
```

**观察不是原因，总分不是解释，相关不是干预效果，运行成功不是结论成立。**


## 课前预测：先暴露自己的判断规则

1. 用一句话回答：没有一个方案全面最好时，怎样做可解释、可复盘的选择？
2. 写出你最可能犯的判断错误，例如“只看平均值”或“看到相关就认定因果”。
3. 为本课写一个可被数据推翻的预测；不要写“应该会更好”这种没有阈值的话。
4. 写出什么结果会让你改变主意。

完成实验后回来修正。保留原答案，因为“怎样改主意”本身就是思维能力证据。


## 一手资料与课程取舍

- [NIST AI RMF：风险、测量与权衡](https://airc.nist.gov/airmf-resources/airmf/5-sec-core/)
- [Model Cards：用途、切片、限制与透明报告](https://research.google/pubs/model-cards-for-model-reporting/)
- [Datasheets for Datasets：数据动机、组成、采集和用途](https://www.microsoft.com/en-us/research/publication/datasheets-for-datasets/)

课程把这些资料转成小型、确定性、可运行的 ASR 实验。示例数据用于理解方法，不代表真实产品结论。


## 1. 先硬约束，再 Pareto，再偏好

1. **硬约束**：隐私、P95 延迟、安全误接受率、内存上限；不满足就淘汰。
2. **Pareto 前沿**：若方案 X 在所有目标不差且至少一项更好，X 支配 Y。
3. **偏好权重**：只在未被支配且满足约束的候选之间表达业务取舍。
4. **敏感性分析**：权重轻微变化就翻转时，决策应标注脆弱，而不是伪装成唯一答案。


In [1]:
candidates = [
    {"name": "tiny-local",  "wer": 14.0, "p95_ms": 90,  "cost": 1.0, "risk": 1.5, "private": True},
    {"name": "base-local",  "wer": 10.5, "p95_ms": 180, "cost": 2.2, "risk": 1.2, "private": True},
    {"name": "large-cloud", "wer": 8.8,  "p95_ms": 420, "cost": 5.5, "risk": 2.8, "private": False},
    {"name": "hybrid",      "wer": 9.6,  "p95_ms": 240, "cost": 3.1, "risk": 1.4, "private": True},
    {"name": "slow-local",  "wer": 11.2, "p95_ms": 310, "cost": 2.8, "risk": 1.8, "private": True},
]

feasible = [c for c in candidates if c["private"] and c["p95_ms"] <= 250]
print("满足硬约束:", [c["name"] for c in feasible])

objectives = ["wer", "p95_ms", "cost", "risk"]  # 全部越低越好
def dominates(a, b):
    return all(a[k] <= b[k] for k in objectives) and any(a[k] < b[k] for k in objectives)

frontier = [c for c in feasible if not any(dominates(other, c) for other in feasible if other is not c)]
print("Pareto 前沿:", [c["name"] for c in frontier])
assert "slow-local" not in frontier


满足硬约束: ['tiny-local', 'base-local', 'hybrid']
Pareto 前沿: ['tiny-local', 'base-local', 'hybrid']


## 2. 加权分数会隐藏价值判断

归一化方式、目标方向和权重都会改变排名。必须把原始指标与硬约束一起展示，不能只发布一个“综合分”。


In [2]:
def minmax_scores(items, weights):
    bounds = {k: (min(x[k] for x in items), max(x[k] for x in items)) for k in weights}
    result = []
    for item in items:
        score = 0.0
        for key, weight in weights.items():
            lo, hi = bounds[key]
            normalized_badness = 0.0 if hi == lo else (item[key] - lo) / (hi - lo)
            score += weight * normalized_badness
        result.append((score, item["name"]))
    return sorted(result)

quality_first = {"wer": .60, "p95_ms": .20, "cost": .10, "risk": .10}
latency_first = {"wer": .25, "p95_ms": .50, "cost": .15, "risk": .10}
print("质量优先:", minmax_scores(frontier, quality_first))
print("延迟优先:", minmax_scores(frontier, latency_first))


质量优先: [(0.2998701298701299, 'base-local'), (0.3666666666666667, 'hybrid'), (0.7, 'tiny-local')]
延迟优先: [(0.35, 'tiny-local'), (0.4368506493506494, 'base-local'), (0.7166666666666667, 'hybrid')]


## 3. 风险不是错误率的同义词

NIST 将风险关联到事件发生可能性与影响。工程中还常显式考虑暴露量：

`risk priority = likelihood × impact × exposure`

它适合排序，不应伪装成精确概率。高风险语音操作还需要安全回退：确认、拒绝、人工接管、审计日志和可回滚版本。


In [3]:
risks = [
    {"name": "音乐请求误识别", "likelihood": .08, "impact": 1, "exposure": 10000},
    {"name": "金额槽位误识别", "likelihood": .01, "impact": 10, "exposure": 1500},
    {"name": "医疗否定词删除", "likelihood": .004, "impact": 10, "exposure": 5000},
]
for item in risks:
    item["priority"] = item["likelihood"] * item["impact"] * item["exposure"]
for item in sorted(risks, key=lambda x: x["priority"], reverse=True):
    print(item["name"], item["priority"])


音乐请求误识别 800.0
医疗否定词删除 200.0
金额槽位误识别 150.0


## 4. 最终产物不是排行榜，而是决策记录

```text
目标场景与非目标场景：
候选与最弱合理基线：
硬约束及来源：
同一固定测试集上的原始指标、切片、区间：
Pareto 前沿：
风险、失败回退、监控与回滚：
选择及被放弃方案：
最强反对意见：
触发重新评估的条件：
```

数据表回答“数据为什么存在、由什么组成、怎样采集、适合/不适合什么”；模型卡回答“模型在什么条件下测过、表现和限制是什么”。两者让后来者能审计今天的选择。


In [4]:
required_decision_fields = {
    "context", "baseline", "constraints", "metrics", "slices", "uncertainty",
    "risks", "fallback", "rollback", "decision", "dissent", "revisit_trigger"
}

def audit_decision_record(record):
    missing = sorted(required_decision_fields - set(record))
    placeholders = {"", "todo", "tbd", "待补充", "未知"}
    empty = sorted(
        key for key in required_decision_fields & set(record)
        if str(record[key]).strip().lower() in placeholders
    )
    return {"missing": missing, "empty": empty, "ready": not missing and not empty}

draft = {key: "TODO" for key in required_decision_fields}
draft["decision"] = "base-local：满足隐私/延迟门禁，质量与成本处于可接受前沿"
print(audit_decision_record(draft))
assert not audit_decision_record(draft)["ready"]


{'missing': [], 'empty': ['baseline', 'constraints', 'context', 'dissent', 'fallback', 'metrics', 'revisit_trigger', 'risks', 'rollback', 'slices', 'uncertainty'], 'ready': False}


## 5. 六课合并成一个思考操作系统

```text
可证伪假设 → 错误分类与切片 → 对照/消融 → 区间与校准
       → 不变量/反事实定位 → 硬约束、Pareto、风险与决策记录
```

遇到新模型时，不先问“是不是更先进”，而依次问：解决哪个可测问题？相比什么基线？在哪些切片？不确定性多大？代价和风险是什么？什么结果会让我改变选择？


## 闭卷挑战

从仓库任一主题中选择两个候选方案，写完整决策记录。至少包含 4 个原始指标、2 个硬约束、3 个风险、Pareto 判断、权重敏感性、失败回退和重新评估触发条件。

回答时强制使用下面的证据卡：

```text
主张：
证据：
最强替代解释：
什么结果会推翻主张：
适用边界：
下一步最小实验：
```


## 最小掌握门禁

- [ ] 我在运行前写了方向和数量级预测。
- [ ] 我能指出示例结论中至少一个替代解释。
- [ ] 我能从空白重写本课核心函数，并用边界输入测试。
- [ ] 我能说明“没有发现差异”和“证明没有差异”的区别。
- [ ] 我把一次被数据推翻的判断写入 `LEARNING_LOG.md`。
- [ ] 我能把本课方法迁移到一个非 ASR 问题。

下一步：回到真实项目，用六课闭环完成一次从问题定义到可审计决策的独立研究。
